# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidism/Machine-Learning-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Finding 1 — "What Predicts Health?" (Random Forest feature importance, ML Appendix)

The paper reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the
top predictors of Health Score, using an 80/20 split.

My methodology question: The dataset spans 57 brands, but the Methodology section doesn't
specify whether the 80/20 split was row-level random or grouped by brand. A random row-level
split could let pages from the same brand appear in both train and test, letting the model
partly learn brand-specific baselines rather than generalizable content signals — the same
issue our own Week-5 model had to correct for with a client-grouped split. I'd also ask: since
Health Score is explicitly built from Position (30 pts) + CTR (20 pts), doesn't predicting
Health Score from Position risk partly measuring the composite formula's own construction
rather than an external relationship? The paper does flag this caveat already, which is the
right instinct — I'd just want the split design stated as explicitly as that caveat.

Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy)

The paper reports Content Age, Days Since Update, and Days Visible as the strongest predictors
of growth, with 71% holdout accuracy.

My methodology question: Same split-design question as Finding 1 — was the 80/20 split
brand-grouped or row-level? Second, on the label itself: if the growth label is computed from
a rolling impression-change window, and "days_visible" is measured over a similar or
overlapping window, there's a risk the feature and label are partially measuring the same
underlying event rather than one genuinely predicting the other. This mirrors the exact
leakage pattern I found in my own Week-3 work when a feature and label shared a time window —
I'd want to know whether days_visible was computed strictly before the growth-window cutoff.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [8]:
from huggingface_hub import login, hf_hub_download
from google.colab import userdata
import pandas as pd
import numpy as np

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

content_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="dim_content.parquet", repo_type="dataset")
perf_path = hf_hub_download(repo_id="FlyRank/internship-warehouse", filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset")

dim_content = pd.read_parquet(content_path)
fact_perf = pd.read_parquet(perf_path)
fact_perf["report_date"] = pd.to_datetime(fact_perf["report_date"])

available = fact_perf[(fact_perf["gsc_data_available"] == True) & (fact_perf["ga4_data_available"] == True)]

page_agg = available.groupby(["content_hash_id","client_hash_id"]).agg(
    total_impressions=("gsc_impressions", "sum"),
    total_clicks=("gsc_clicks", "sum"),
    avg_position=("gsc_avg_position", "mean")
).reset_index()
page_agg["ctr"] = page_agg["total_clicks"] / page_agg["total_impressions"].replace(0, pd.NA)

df = page_agg.merge(dim_content[["content_hash_id","word_count","char_count","backlinks","search_volume"]],
                     on="content_hash_id", how="left")
df = df.dropna(subset=["avg_position","ctr","word_count","char_count","backlinks","search_volume"])
df = df[df["total_impressions"] >= 50]

df["position_bucket"] = pd.cut(df["avg_position"], bins=[0,3,10,20,50,1000], labels=["1-3","4-10","11-20","21-50","50+"])
df["expected_ctr"] = df.groupby("position_bucket", observed=True)["ctr"].transform("mean")
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]

first_half = fact_perf[fact_perf["report_date"] < "2026-03-16"]
second_half = fact_perf[fact_perf["report_date"] >= "2026-03-16"]
imp_first = first_half.groupby("content_hash_id")["gsc_impressions"].sum()
imp_second = second_half.groupby("content_hash_id")["gsc_impressions"].sum()
trend = pd.DataFrame({"imp_first": imp_first, "imp_second": imp_second}).dropna()
trend["label_down"] = (trend["imp_second"] < trend["imp_first"]).astype(int)

df = df.merge(trend[["label_down"]], left_on="content_hash_id", right_index=True, how="inner")
print("Dataset ready:", df.shape)

Dataset ready: (26562, 14)


## 2. My model under an honest split (before/after)
Before (naive random split): 20 clients appeared in BOTH train and test sets. This inflated
precision dramatically — 0.95 at k=20, 0.90 at k=50, 0.88 at k=100 — because the model could
partly memorize client-specific patterns rather than learning generalizable content signals.

After (honest client-grouped split, same as Week 5): 0 client overlap. Precision dropped to
0.60 at k=20, 0.64 at k=50, 0.70 at k=100.

The gap is large — roughly 18 to 35 percentage points depending on k — which shows just how
misleading an ungrouped split can be on this dataset. This is the exact concern I raised about
the paper's own 80/20 splits in Section 1: without knowing whether those splits were grouped
by brand, a reported accuracy could be inflated in a very

In [9]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

feature_cols = ["word_count","char_count","backlinks","avg_position","ctr_gap","search_volume"]
X = df[feature_cols]
y = df["label_down"]
groups = df["client_hash_id"]

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)
    top_k = order[:k]
    return y_true.values[top_k].mean()

# BEFORE: naive random split
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.3, random_state=42)
rf_random = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf_random.fit(X_train_r, y_train_r)
scores_random = rf_random.predict_proba(X_test_r)[:, 1]

# AFTER: client-grouped split (honest, same as Week 5)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, class_weight="balanced")
rf_grouped.fit(X_train_g, y_train_g)
scores_grouped = rf_grouped.predict_proba(X_test_g)[:, 1]

overlap_random = len(set(groups.loc[X_train_r.index]) & set(groups.loc[X_test_r.index]))
overlap_grouped = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

print("Client overlap — naive random split:", overlap_random)
print("Client overlap — grouped split:", overlap_grouped)
print()

comparison = []
for k in (20, 50, 100):
    comparison.append({
        "k": k,
        "naive_random_precision": round(precision_at_k(scores_random, y_test_r, k), 3),
        "honest_grouped_precision": round(precision_at_k(scores_grouped, y_test_g, k), 3)
    })
print(pd.DataFrame(comparison))

Client overlap — naive random split: 20
Client overlap — grouped split: 0

     k  naive_random_precision  honest_grouped_precision
0   20                    0.95                      0.60
1   50                    0.90                      0.64
2  100                    0.88                      0.70


## 3. Leakage audit

Leakage audit finding: word_count, char_count, backlinks, and search_volume are static content
properties — no time-leak risk. However, avg_position and ctr_gap are computed from the FULL
March window, which technically overlaps with the second half of March — the same window used
to compute the label (label_down compares first-half vs second-half impressions).

To quantify the actual risk, I compared the correlation of the full-month ctr_gap with the
label (0.067, weak) against a first-half-only CTR proxy's correlation with the label (-0.153,
moderate). The full-month version does NOT show a stronger correlation than the first-half-only
version — if anything it's weaker — which suggests the theoretical overlap isn't translating
into meaningful leakage in practice, unlike the deliberate leak I built in Week 3 where adding
a label-derived column pushed the correlation dramatically higher.

That said, this is a genuine methodological gap I'm flagging rather than dismissing: the
correct fix for a production version of this pipeline would be to compute all features
(including avg_position and ctr_gap) using ONLY the first half of March, reserving the second
half exclusively for constructing the label. I did not rebuild the full pipeline with that
stricter cut for this audit, but the check above gives me reasonable confidence the current
leakage risk is small, not the more shown "you get to work" jump.

In [10]:
for col in feature_cols:
    print(f"{col}: {'STATIC (no time-leak risk)' if col in ['word_count','char_count','backlinks','search_volume'] else 'DERIVED FROM MARCH PERFORMANCE — verify window'}")

print("\nctr_gap uses avg_position from the FULL month (includes second half, same window as the label).")

first_half_agg = first_half[first_half["content_hash_id"].isin(df["content_hash_id"])].groupby("content_hash_id").agg(
    fh_impressions=("gsc_impressions","sum"), fh_clicks=("gsc_clicks","sum"), fh_avg_position=("gsc_avg_position","mean")
).reset_index()
first_half_agg["fh_ctr"] = first_half_agg["fh_clicks"] / first_half_agg["fh_impressions"].replace(0, pd.NA)

df_check = df.merge(first_half_agg, on="content_hash_id", how="inner")
print("\nCorrelation: full-month ctr_gap vs label:", df_check["ctr_gap"].corr(df_check["label_down"]))
print("Correlation: first-half-only ctr (proxy) vs label:", df_check["fh_ctr"].corr(df_check["label_down"]))

word_count: STATIC (no time-leak risk)
char_count: STATIC (no time-leak risk)
backlinks: STATIC (no time-leak risk)
avg_position: DERIVED FROM MARCH PERFORMANCE — verify window
ctr_gap: DERIVED FROM MARCH PERFORMANCE — verify window
search_volume: STATIC (no time-leak risk)

ctr_gap uses avg_position from the FULL month (includes second half, same window as the label).

Correlation: full-month ctr_gap vs label: 0.06707846896513087
Correlation: first-half-only ctr (proxy) vs label: -0.1526835083716464


## 4. Claim rewrite

My boldest original claim (from Week 5): "The model outperforms the baseline at every k,
confirming the additional features carry real predictive signal beyond ctr_gap alone."

Rewritten in safe language: On a client-grouped test split, the Random Forest model showed
higher measured Precision@20/50/100 (0.60/0.64/0.70) than the ctr_gap-only baseline. This is
an observed, directional result on one mid-panel month (March 2026) of one dataset slice — it
supports using the model for decision-support in prioritizing review candidates, but it is not
proof the features will outperform the baseline on future months, other clients, or at
deployment. This week's audit reinforced why the "honest" qualifier matters: a naive random
split on this exact same data inflated precision to 0.95/0.90/0.88 — nearly the ceiling —
purely from client leakage, with zero change to the features or model itself. That gap alone
is a reminder to keep re-validating this comparison as new data arrives, rather than treating
one month's grouped-split result as a settled, permanent finding.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.